# Databricks data ingestion
There are three methods that could be used to ingest data from files stored in the cloud storage.
- CREATE TABLE AS (CTAS) : 
    - It creates a delta table by default from files stored in the cloud object storage.
    - The ```read_files()``` function is used to read files from a specified loaction and return the data in a tabular format.
    - It offers several capabilities:
        - Supports various file formats like JSON, csv, xml, text, binaryfilem paraquet, avro and orc
        - Automatically detects file format and infers a unified schema across all files.
        - Allows you to specify format-specific options for greater control when reading source files.
        - Can be used in streaming tables to incrementally ingest files into delta lake using auto loader. We will learn more about auto loader shortly. 
- COPY INTO : 
    - This is used to copy files from cloud storage into the delta table. 
    - This command performs bulk load from files in cloud object storage into the table, and in this example, it will load files into the empty table new_table.
    - The FROM clause specifies the location of the csv files
    - You start by creating a table which can be defined with or without a schema. In this case, we will create a table named new_table without a schema.
    - COPY INTO is ideal for situations where the cloud storage location is continuously adding files, since it is a reltriable and indepodent operation designed for incremental batch ingestion.
        - What that means is : COPY INTO will skip any files that have already been loaded into the table, and only new files will be ingested. 
        - Now lets go over some of the key aspects of COPY INTO: 
            - It supports various common file types liek parquet, JSON, XML and others.
            - The FROM clause specifies the path of the cloud storage where new files are being continously added.
            - FORMAT_OPTIONS{} controls how the source files are parsed and interpreted and the available options will depend on the file format you are working with.
            - COPY_OPTIONS() lets you control the behaviour of COPY INTO operation itself. For example: options like schema evolution using mergeSchema, or idempotency using force.
- Auto Loader : 
    - Incrementally and efficiently process new data files as they arrive in cloud storage without any additional setup.
    - Auto loader has support for both python and sql (leveraging declarative pipelines)
    - You can use Auto Loader to process billions of files. 
    - Auto loader is built upon **Spark Structured Streaming**

## Schema : 
- A schema (in terms of databases) is a formal language which describes the structure of data (blueprint) of a database.
- A schema can define many different data structures that serve different purpose for a database.
- Different data structures (relational databases):
    - Tables
    - Fields
    - Views
    - Relationships
    - Indexes
    - Packages
    - Procedures
    - Functions
    - XML schemas
    - Queues
    - Triggers
    - Types
    - Sequences
    - materialized views
    - Synonyms
    - database links
    - Directories
## Schemaless : 
- Schemaless is when the primary "cell" of database can accept many types.
- This allows developers to forgo the upfront data modelling.
- Common schemaless databases are  : 
    - Key/Value
    - Document
    - Columns
        - Wide column
    - graph

## Data Documents
- A data document defines the collective form in which data exists.
- Common types of data documents : 
    - Datasets : a logical grouping of data
    - Databases : structured data that can be quickly accessed and searched
    - Datastores : unstructured or semi-structured data to housing data
    - Data warehouse : structured or semi-structured data for creating reports and analytics.
    - Notebooks : data that is arranged in pages, designed for easy consumption

## Data sets
- A data sets is a logical grouping of units of data that generally are closely related and/or share the same data structure.
- Just because I said data structure doesn't always mean that the data itself is structured, it can be a semi-structured or un-structured data
- There are publically available data sets that are used in the learning of statistics, data analytics, machine learning
- MNIST database Images of handwritten digits used to test classification, clustering and image processing algorithms.
- Commonly used when learning how to build computer vision ML models to translate handwriting into digital text.
- COCO dataset (Common objects in Context dataset) : A dataset which contains many common images using a JSON file (coco format) that identify objects or segments within an image.
- IMDB riviews datasets : A movie dataset with 25,000 highly popular movie reviews for training and 25000 for testing.


## Query and Querying 
- A query is a request for data results (reads) or to perform operations such as inserting updating deleting data (writes).
- A query can perform maintanance operations on the data and is not always restricted to just working with the data that resides the database.
- Querying : This is an act of performing a query
- what is a query language ? : A scripting language designed as the format to submit a request or actions to the database. Notable query languages: 
    - SQL
    - GraphSQL
    - Kusto
    - Xpath
    - Gremlin

## Batch vs Stream processing
### Batch processing
![batch_processing](images/batch_processing.png)
- When you send batches (a collection) of data to be processed. 
- Batches are generally scheduled example : Every day at 1PM
- Batches are not real-time.
- Batche processing is ideal for very large processing workloads.
- Batch processing is more cost-effective
### Stream processing
![stream_processing](images/stream_processing.png)
When you process data as soon as it arrives : 
- Produces will send data to a stream 
- Consumers will pull from the stream
- A data can be held in a stream for a period of time so that we can have a better re-usability of data.
- It is suitable for real-time processing (streaming-video)
- Much more expensive than batch processing.


## Pivot table
- A pivot table is a table of statistics that summarizes the data of a more extensive table from a : Database, Spreadsheet or Business intelligence (BI) tool
- Pivot tables are a technique in data processing.
- They arrange and rearrange (or "Pivot") statistics in order to draw attention to useful information
- This leads to finding figures and facts quickly making them integral to data analysis.
- Here is the example table :
| Region | Product | Sales |
| ------ | ------- | ----- |
| East   | Apples  | 100   |
| East   | Bananas | 150   |
| West   | Apples  | 200   |
| West   | Bananas | 120   |
- Here is the pivot table of the above table : 
| Region    | Apples  | Bananas | Total   |
| --------- | ------- | ------- | ------- |
| East      | 100     | 150     | 250     |
| West      | 200     | 120     | 320     |
| **Total** | **300** | **270** | **570** |
- Here is a sample code for generating a pivot table from a table (python > Pandas)
```python
import pandas as pd

data = {
    "Region": ["East", "East", "West", "West"],
    "Product": ["Apples", "Bananas", "Apples", "Bananas"],
    "Sales": [100, 150, 200, 120],
}
df = pd.DataFrame(data)

pivot = df.pivot_table(index="Region", columns="Product", values="Sales", aggfunc="sum")
print(pivot)
```
- Here is a sample code for generating a pivot table from a table in (pyspark)
```python
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum

spark = SparkSession.builder.getOrCreate()

data = [("East", "Apples", 100),
        ("East", "Bananas", 150),
        ("West", "Apples", 200),
        ("West", "Bananas", 120)]

df = spark.createDataFrame(data, ["Region", "Product", "Sales"])

pivot_df = df.groupBy("Region").pivot("Product").agg(sum("Sales"))
pivot_df.show()
```
- **When to use pivot table:**
    - Summarize large datasets quickly
    - Find totals, averages or counts grouped by categories.
    - compare data across multiple dimensions (example region vs products)
    - Explore patterns or trends in your dataset.
    
